In [ ]:
# Prerequisites
! pip install vietocr
! pip install "paddlex[ocr]"
! pip install paddlepaddle-gpu==3.0.0 -i https://www.paddlepaddle.org.cn/packages/stable/cu126/
! pip install torch torchvision --index-url https://download.pytorch.org/whl/cu126

In [1]:
import matplotlib.pyplot as plt
from PIL import Image

from vietocr.tool.predictor import Predictor
from vietocr.tool.config import Cfg

In [2]:
config = Cfg.load_config_from_name('vgg_seq2seq')
config['cnn']['pretrained']=False
config['device'] = 'cuda:0'

In [3]:
detector = Predictor(config)

Model weight /tmp/vgg_seq2seq.pth exsits. Ignore download!


In [4]:
from paddlex import create_model
text_det = create_model("PP-OCRv5_server_det")


/media/bob-in-linux/freespace/Coding-Space/paddlex_vietocr/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Checking connectivity to the model hosters, this may take a while. To bypass this check, set `PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK` to `True`.
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/home/bob-in-linux/.paddlex/official_models/PP-OCRv5_server_det`.
/media/bob-in-linux/freespace/Coding-Space/paddlex_vietocr/.venv/lib/python3.12/site-packages/paddle/utils/cpp_extension/extension_utils.py:711: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)


In [5]:
# RUN FROM HERE AFTER REPLACING THE INPUT IMAGE
output = text_det.predict(input="./test_img/vnese_test1_cropped.png")

In [6]:

bounding_coords = []
for res in output:
    bounding_coords = res['dt_polys']
    res.print()
    res.save_to_img(save_path="./output/")
    res.save_to_json(save_path="./output/")


{'res': {'input_path': './test_img/vnese_test1_cropped.png', 'page_index': None, 'dt_polys': array([[[ 41, 601],
        ...,
        [ 41, 638]],

       ...,

       [[158,   3],
        ...,
        [157,  41]]], shape=(18, 4, 2), dtype=int16), 'dt_scores': [0.8176938169693256, 0.8697566227098608, 0.8565957164536345, 0.9478769008117245, 0.8559496527218531, 0.8795035392968644, 0.8791705973256329, 0.8918183256301873, 0.8525202576960241, 0.8420163548196592, 0.8613208840172147, 0.869929877420521, 0.8131983413404653, 0.764329473417079, 0.7702355514270157, 0.7408402898841809, 0.8840921830501781, 0.8768824089817744]}}


In [7]:
# These coords follows the format of 4 
bounding_coords

array([[[ 41, 601],
        ...,
        [ 41, 638]],

       ...,

       [[158,   3],
        ...,
        [157,  41]]], shape=(18, 4, 2), dtype=int16)

In [8]:
bounding_coords_rect = []
for coords in bounding_coords:
    x_min = min(coords[0][0], coords[1][0], coords[2][0], coords[3][0])
    y_min = min(coords[0][1], coords[1][1], coords[2][1], coords[3][1])
    x_max = max(coords[0][0], coords[1][0], coords[2][0], coords[3][0])
    y_max = max(coords[0][1], coords[1][1], coords[2][1], coords[3][1])
    bounding_coords_rect.append([x_min, y_min, x_max, y_max])
    
bounding_coords_rect

[[np.int16(41), np.int16(594), np.int16(545), np.int16(638)],
 [np.int16(43), np.int16(564), np.int16(544), np.int16(603)],
 [np.int16(42), np.int16(531), np.int16(545), np.int16(568)],
 [np.int16(48), np.int16(497), np.int16(222), np.int16(525)],
 [np.int16(47), np.int16(461), np.int16(542), np.int16(493)],
 [np.int16(50), np.int16(430), np.int16(543), np.int16(461)],
 [np.int16(51), np.int16(398), np.int16(542), np.int16(430)],
 [np.int16(53), np.int16(367), np.int16(543), np.int16(400)],
 [np.int16(55), np.int16(335), np.int16(542), np.int16(369)],
 [np.int16(56), np.int16(303), np.int16(542), np.int16(339)],
 [np.int16(28), np.int16(270), np.int16(543), np.int16(310)],
 [np.int16(6), np.int16(232), np.int16(129), np.int16(267)],
 [np.int16(8), np.int16(202), np.int16(547), np.int16(244)],
 [np.int16(9), np.int16(169), np.int16(549), np.int16(215)],
 [np.int16(13), np.int16(140), np.int16(550), np.int16(184)],
 [np.int16(37), np.int16(106), np.int16(551), np.int16(156)],
 [np.int16(

In [9]:
bounding_coords_rect = sorted(bounding_coords_rect, key=lambda x: x[1])

In [10]:
img = './test_img/vnese_test1_cropped.png'
img = Image.open(img)
result_strings = []
for coord in bounding_coords_rect:
    cropped = img.crop(coord)
    s = detector.predict(cropped)
    result_strings.append(s)
    
result_strings

['I: Dấu hiệu chung',
 'về nguy cơ bạo lực tiềm tàng',
 'Ngoài các phân tích tâm lý đã đề cập tới trong',
 'các phần trước của chương này, bạn cũng có thể',
 'nhìn vào một số dấu hiệu sau đây. để biết liệu',
 'một người có biểu hiện nào của bạo lực tiềm',
 'tàng không:',
 'Người đó kể về thời thơ ấu, bố mẹ, anh chị em',
 'họ hàng, bạn bè thuở bé của họ thế nào? Một',
 'người nếu nói một cách cay nghiệt về thời thơ ấu',
 'hay họ hàng của anh ta, sử dụng những ngôn từ',
 'mạnh, thậm chí đôi khi là ngôn ngữ bạo lực thì',
 'chắc chắn anh ta có những vấn đề trong quá khứ',
 'vẫn chưa được giải quyết, rất có thể dẫn tới hậu',
 'quả khôn lường',
 'Liệu người đó có từng bị lạm dụng? Robert',
 'Ressler, chuyên gia phân tích tâm lý tội phạm',
 "của FBI, người đã tạo ra thuật ngữ 'kẻ sát nhân"]